# 订单事实与指标口径审计

步骤 8 的可复现审计伴侣：只读 MySQL Mart 与 Ops 审计表。

## tl;dr

构建运行 `cfd9f615-d65a-4113-a8d4-c0bbe2e1317f` 已创建 19,960 个 UCI 订单和 189 个 GA4 规范订单。GA4 的 435 条 `(not set)` 交易号事件被排除在订单事实之外；UCI GBP 与 GA4 USD 不能相加。

## Context & Methods

### Key Assumptions

- UCI：一张有效发票为一个订单，只使用 `is_valid_sale_line = 1` 的明细。
- GA4：有意义的 `transaction_id` 为一个订单；`(not set)` 和 `unknown` 不是订单键。
- 同一 GA4 交易号出现多次时，按 `event_ts, stg_event_id` 保留最早事件。
- UCI 金额单位 GBP，GA4 金额单位 USD，结果只能并列展示。

## Data

来源批次：UCI `7aa2b0f4-12aa-4dfb-890c-ffadc8f9d6fa`；GA4 `9a2d8d7c-0f5b-4c57-a54d-98c59b9c5a27`。完整定义见 `docs/metric_dictionary.md`。

In [1]:
from pathlib import Path
import os

import pandas as pd
import pymysql
from dotenv import load_dotenv

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(project_root / '.env')
build_run_id = 'cfd9f615-d65a-4113-a8d4-c0bbe2e1317f'

connection = pymysql.connect(
    host=os.environ.get('MYSQL_HOST', '127.0.0.1'),
    port=int(os.environ.get('MYSQL_PORT', '3306')),
    user=os.environ.get('MYSQL_USER', 'root'),
    password=os.environ['MYSQL_ROOT_PASSWORD'],
    charset='utf8mb4',
)

## Results

In [2]:
run_audit = pd.read_sql(
    '''SELECT run_id, status, definition_version, rows_written, input_description, started_at, finished_at
       FROM insightops_ops.transformation_run WHERE run_id = %s''',
    connection, params=(build_run_id,)
)
run_audit

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_4078/1282435306.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  run_audit = pd.read_sql(


,run_id,status,definition_version,rows_written,input_description,started_at,finished_at
0,cfd9f615-d65a-4113-a8d4-c0bbe2e1317f,SUCCEEDED,order_facts_v1,20149,"{""ga4_batch_id"": ""9a2d8d7c-0f5b-4c57-a54d-98c5...",2026-08-23 13:56:23.254121,2026-08-23 05:56:26.938507


In [3]:
order_metrics = pd.read_sql(
    '''SELECT 'UCI (GBP)' AS source, COUNT(*) AS canonical_orders,
              ROUND(SUM(gmv_gbp), 2) AS canonical_revenue, COUNT(DISTINCT customer_id) AS distinct_buyers
       FROM insightops_mart.uci_order_fact WHERE build_run_id = %s
       UNION ALL
       SELECT 'GA4 (USD)', COUNT(*), ROUND(SUM(revenue_usd), 2), COUNT(DISTINCT user_pseudo_id)
       FROM insightops_mart.ga4_order_fact WHERE build_run_id = %s''',
    connection, params=(build_run_id, build_run_id)
)
order_metrics

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_4078/445084423.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  order_metrics = pd.read_sql(


,source,canonical_orders,canonical_revenue,distinct_buyers
0,UCI (GBP),19960,10642110.8,4338
1,GA4 (USD),189,12836.0,170


In [4]:
exclusion_audit = pd.read_sql(
    '''SELECT source_name, input_rows, excluded_rows, output_rows, notes
       FROM insightops_ops.fact_build_audit WHERE run_id = %s ORDER BY source_name''',
    connection, params=(build_run_id,)
)
exclusion_audit

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_4078/480565745.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  exclusion_audit = pd.read_sql(


,source_name,input_rows,excluded_rows,output_rows,notes
0,GA4,625,435,189,排除占位交易号；同一交易号以最早事件保留一条规范订单。
1,UCI,524878,0,19960,仅聚合 is_valid_sale_line = 1；客户 ID 缺失不排除订单。


## Takeaways

下游漏斗、客户价值与异常检测读取这两个事实表，而非直接汇总 Raw 或 Staging。任何跨源比较都必须保留来源标签和货币单位。